# SPX Diagonal Calendar - Trade Forensics
## Paper Trade: June 23, 2026

| Leg | Expiry | Strike | Fill |
|-----|--------|--------|------|
| STO Call | 6/26 | 7500 | 11.70 |
| STO Put  | 6/26 | 7300 | 19.30 |
| BTO Call | 6/29 | 7500 | 18.00 |
| BTO Put  | 6/29 | 7300 | 26.50 |
| **Net Debit** | | | **13.50** |

**Transformation at 3:50 PM: Net Credit 19.85 | Locked P&L +$635 | Max Loss $135**

## Cell 1 - Configuration
Edit `DB_PATH` to point at your `dashboard.db`. Everything else is pre-filled.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from datetime import date
import warnings
warnings.filterwarnings('ignore')

# ── EDIT THIS ──────────────────────────────────────────────────────────────────
DB_PATH = r"C:\Users\chand\Python\spx-diagonal-dashboard\data\dashboard.db"   # <-- update to actual path location

# ── TRADE FILLS (do not change) ────────────────────────────────────────────────
ENTRY_BACK_CALL   = 18.00
ENTRY_BACK_PUT    = 26.50
ENTRY_FRONT_CALL  = 11.70
ENTRY_FRONT_PUT   = 19.30
ENTRY_NET_DEBIT   = 13.50
TRANSFORM_CREDIT  = 19.85
LOCKED_PNL        = TRANSFORM_CREDIT - ENTRY_NET_DEBIT

FRONT_EXPIRY     = "2026-06-26"
BACK_EXPIRY      = "2026-06-29"
CALL_STRIKE      = 7500.0
PUT_STRIKE       = 7300.0

# Timestamps in DB are UTC. 3:30 PM ET = 19:30 UTC, 4:05 PM ET = 20:05 UTC
WINDOW_START   = "2026-06-23 19:30:00"
WINDOW_END     = "2026-06-23 20:05:00"
TRANSFORM_TIME = pd.Timestamp("2026-06-23 19:50:00")

PROFIT_THRESHOLD   = 5.00
IV_RATIO_THRESHOLD = 1.15

print("Config loaded.")
print(f"  DB_PATH         : {DB_PATH}")
print(f"  Entry debit     : ${ENTRY_NET_DEBIT}")
print(f"  Transform credit: ${TRANSFORM_CREDIT}")
print(f"  Locked P&L      : +${LOCKED_PNL:.2f}")
print(f"  Window (UTC)    : {WINDOW_START} to {WINDOW_END}")


Config loaded.
  DB_PATH         : C:\Users\chand\Python\spx-diagonal-dashboard\data\dashboard.db
  Entry debit     : $13.5
  Transform credit: $19.85
  Locked P&L      : +$6.35
  Window (UTC)    : 2026-06-23 19:30:00 to 2026-06-23 20:05:00


## Cell 2 - Schema Verification & Snapshot Gap Check
Confirms DB column names, then checks for missing minutes in the 3:30-4:00 PM window.

In [ ]:
conn = sqlite3.connect(DB_PATH)

print("=== Actual column names ===")
for tbl in ["snapshots", "option_rows", "atm_iv_by_expiry", "collection_gaps"]:
    cur = conn.execute(f"PRAGMA table_info({tbl})")
    cols = [r[1] for r in cur.fetchall()]
    print(f"  {tbl}: {cols}")

gap_sql = """
SELECT
    snapshot_id,
    snapshot_timestamp,
    status,
    market_session,
    LAG(snapshot_timestamp) OVER (ORDER BY snapshot_timestamp) AS prev_ts,
    ROUND(
        (julianday(snapshot_timestamp)
         - julianday(LAG(snapshot_timestamp) OVER (ORDER BY snapshot_timestamp))
        ) * 1440, 1
    ) AS minutes_since_last
FROM snapshots
WHERE snapshot_timestamp BETWEEN ? AND ?
ORDER BY snapshot_timestamp
"""

df_snaps = pd.read_sql_query(gap_sql, conn, params=(WINDOW_START, WINDOW_END))
df_snaps["snapshot_timestamp"] = pd.to_datetime(df_snaps["snapshot_timestamp"])

total   = len(df_snaps)
complete= (df_snaps["status"] == "COMPLETE").sum()
partial = (df_snaps["status"] == "PARTIAL").sum()
gaps    = df_snaps[df_snaps["minutes_since_last"] > 1.5]

print(f"\n=== Snapshot summary ===")
print(f"  Total in window : {total}")
print(f"  COMPLETE        : {complete}")
print(f"  PARTIAL         : {partial}")
print(f"  Gaps > 90 sec   : {len(gaps)}")

if len(gaps):
    print("
GAPS DETECTED:")
    print(gaps[["snapshot_timestamp","status","minutes_since_last"]].to_string(index=False))
else:
    print("
No gaps - clean 1-minute coverage.")

print("
=== All snapshots in window ===")
print(df_snaps[["snapshot_id","snapshot_timestamp","status",
                "market_session","minutes_since_last"]].to_string(index=False))


SyntaxError: unterminated f-string literal (detected at line 34) (1195563038.py, line 34)

## Cell 3 - Pull Option Chain Data
Pulls the 4 core trade legs plus neighboring strikes. Note: `underlying_price` is on `snapshots`; `mark` is the pre-computed mid on `option_rows`.

In [ ]:
option_sql = """
SELECT
    s.snapshot_id,
    s.snapshot_timestamp,
    s.underlying_price,
    s.vix_value,
    o.expiry_date,
    o.dte,
    o.strike,
    o.right,
    o.bid,
    o.ask,
    o.mark,
    o.iv,
    o.delta,
    o.gamma,
    o.theta,
    o.vega,
    o.intrinsic_value,
    o.time_value,
    o.volume,
    o.open_interest
FROM option_rows o
JOIN snapshots s ON o.snapshot_id = s.snapshot_id
WHERE
    s.status IN ('COMPLETE', 'PARTIAL')
    AND s.snapshot_timestamp BETWEEN ? AND ?
    AND o.expiry_date IN (?, ?)
    AND (
        (UPPER(o.right) IN ('C','CALL') AND o.strike IN (7495.0, 7500.0, 7505.0))
        OR
        (UPPER(o.right) IN ('P','PUT')  AND o.strike IN (7295.0, 7300.0, 7305.0))
    )
ORDER BY s.snapshot_timestamp, o.expiry_date, o.right, o.strike
"""

df_raw = pd.read_sql_query(
    option_sql, conn,
    params=(WINDOW_START, WINDOW_END, FRONT_EXPIRY, BACK_EXPIRY)
)
df_raw["snapshot_timestamp"] = pd.to_datetime(df_raw["snapshot_timestamp"])
df_raw["right"] = df_raw["right"].str.upper().str[0]   # normalize to C / P

print(f"Rows fetched       : {len(df_raw)}")
print(f"Unique timestamps  : {df_raw['snapshot_timestamp'].nunique()}")
print(f"Unique right values: {df_raw['right'].unique().tolist()}")
print("
Contracts present:")
summary = (df_raw
    .groupby(["expiry_date","right","strike"])
    .agg(snapshots=("mark","count"),
         mark_min=("mark","min"), mark_max=("mark","max"),
         iv_min=("iv","min"),   iv_max=("iv","max"))
    .reset_index())
print(summary.to_string(index=False))


## Cell 4 - Reconstruct Diagonal Mark Minute by Minute

`Diagonal Mark = (Back Call + Back Put) - (Front Call + Front Put)`

`Unrealized P&L = Mark - $13.50 entry cost`

Data starts at 3:30 PM; entry fills at 1:46 PM are the fixed baseline.

In [ ]:
def is_core(row):
    r, ex, st = row["right"], row["expiry_date"], row["strike"]
    return ((ex == BACK_EXPIRY  and r == "C" and st == CALL_STRIKE) or
            (ex == BACK_EXPIRY  and r == "P" and st == PUT_STRIKE)  or
            (ex == FRONT_EXPIRY and r == "C" and st == CALL_STRIKE) or
            (ex == FRONT_EXPIRY and r == "P" and st == PUT_STRIKE))

core = df_raw[df_raw.apply(is_core, axis=1)].copy()
core["leg"] = (core["expiry_date"].str[-5:] + "_" +
               core["right"] + "_" +
               core["strike"].astype(int).astype(str))

piv_mark = core.pivot_table(
    index=["snapshot_timestamp","underlying_price"],
    columns="leg", values="mark", aggfunc="first"
).reset_index()

piv_iv = core.pivot_table(
    index="snapshot_timestamp",
    columns="leg", values="iv", aggfunc="first"
).reset_index()
piv_iv.columns = ["snapshot_timestamp"] + ["iv_" + c for c in piv_iv.columns[1:]]

df_m = piv_mark.merge(piv_iv, on="snapshot_timestamp", how="left")
df_m = df_m.sort_values("snapshot_timestamp").reset_index(drop=True)

print("Available mark columns:")
mark_cols = [c for c in df_m.columns
             if c not in ["snapshot_timestamp","underlying_price"]
             and not c.startswith("iv_")]
print(" ", mark_cols)

def find_col(df, expiry_tail, right, strike, prefix=""):
    pat = f"{prefix}{expiry_tail}_{right}_{int(strike)}"
    matches = [c for c in df.columns if pat in c]
    return matches[0] if matches else None

bc = find_col(df_m, "06-29", "C", CALL_STRIKE)
bp = find_col(df_m, "06-29", "P", PUT_STRIKE)
fc = find_col(df_m, "06-26", "C", CALL_STRIKE)
fp = find_col(df_m, "06-26", "P", PUT_STRIKE)
bc_iv = find_col(df_m, "06-29", "C", CALL_STRIKE, "iv_")
bp_iv = find_col(df_m, "06-29", "P", PUT_STRIKE,  "iv_")
fc_iv = find_col(df_m, "06-26", "C", CALL_STRIKE, "iv_")
fp_iv = find_col(df_m, "06-26", "P", PUT_STRIKE,  "iv_")

print(f"
Column map:")
print(f"  back_call mark={bc}  iv={bc_iv}")
print(f"  back_put  mark={bp}  iv={bp_iv}")
print(f"  front_call mark={fc} iv={fc_iv}")
print(f"  front_put  mark={fp} iv={fp_iv}")

if None in [bc, bp, fc, fp]:
    print("
ERROR: Could not find all 4 leg columns. Check marks above.")
else:
    df_m["back_call_mark"]  = df_m[bc]
    df_m["back_put_mark"]   = df_m[bp]
    df_m["front_call_mark"] = df_m[fc]
    df_m["front_put_mark"]  = df_m[fp]
    df_m["diagonal_mark"]   = (df_m["back_call_mark"] + df_m["back_put_mark"]
                                - df_m["front_call_mark"] - df_m["front_put_mark"])
    df_m["unrealized_pnl"]     = df_m["diagonal_mark"] - ENTRY_NET_DEBIT
    df_m["unrealized_pnl_pct"] = df_m["unrealized_pnl"] / ENTRY_NET_DEBIT * 100

    for col, src in [("back_call_iv",bc_iv),("back_put_iv",bp_iv),
                     ("front_call_iv",fc_iv),("front_put_iv",fp_iv)]:
        df_m[col] = df_m[src] if src and src in df_m.columns else np.nan

    df_m["call_iv_spread"] = df_m["back_call_iv"] - df_m["front_call_iv"]
    df_m["put_iv_spread"]  = df_m["back_put_iv"]  - df_m["front_put_iv"]
    df_m["call_iv_ratio"]  = df_m["back_call_iv"] / df_m["front_call_iv"].replace(0, np.nan)
    df_m["put_iv_ratio"]   = df_m["back_put_iv"]  / df_m["front_put_iv"].replace(0, np.nan)

    disp = ["snapshot_timestamp","underlying_price",
            "diagonal_mark","unrealized_pnl","unrealized_pnl_pct",
            "back_call_mark","back_put_mark","front_call_mark","front_put_mark",
            "call_iv_spread","put_iv_spread","call_iv_ratio","put_iv_ratio"]
    pd.set_option("display.float_format", "{:.4f}".format)
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 220)
    print("
Diagonal Mark - Minute by Minute")
    print("=" * 120)
    print(df_m[[c for c in disp if c in df_m.columns]].to_string(index=False))


## Cell 5 - Time Value & Calendar Edge

The collector pre-computes `time_value` per contract. Calendar edge = back time value - front time value at the same strike. A shrinking edge signals the structure is losing its advantage.

In [ ]:
tv_sql = """
SELECT
    s.snapshot_timestamp,
    o.expiry_date,
    o.right,
    o.strike,
    o.mark,
    o.time_value,
    o.intrinsic_value,
    o.iv
FROM option_rows o
JOIN snapshots s ON o.snapshot_id = s.snapshot_id
WHERE
    s.status IN ('COMPLETE','PARTIAL')
    AND s.snapshot_timestamp BETWEEN ? AND ?
    AND o.expiry_date IN (?, ?)
    AND (
        (UPPER(o.right) IN ('C','CALL') AND o.strike = ?)
        OR
        (UPPER(o.right) IN ('P','PUT')  AND o.strike = ?)
    )
ORDER BY s.snapshot_timestamp, o.expiry_date, o.right
"""

df_tv = pd.read_sql_query(
    tv_sql, conn,
    params=(WINDOW_START, WINDOW_END, FRONT_EXPIRY, BACK_EXPIRY,
            CALL_STRIKE, PUT_STRIKE)
)
df_tv["snapshot_timestamp"] = pd.to_datetime(df_tv["snapshot_timestamp"])
df_tv["right"] = df_tv["right"].str.upper().str[0]
df_tv["leg"] = (df_tv["expiry_date"].str[-5:] + "_" + df_tv["right"] + "_" +
                df_tv["strike"].astype(int).astype(str))

piv_tv = df_tv.pivot_table(
    index="snapshot_timestamp", columns="leg",
    values="time_value", aggfunc="first"
).reset_index()
piv_tv.columns.name = None

bc_tv = [c for c in piv_tv.columns if "06-29_C" in c]
bp_tv = [c for c in piv_tv.columns if "06-29_P" in c]
fc_tv = [c for c in piv_tv.columns if "06-26_C" in c]
fp_tv = [c for c in piv_tv.columns if "06-26_P" in c]

df_edge = piv_tv.copy()
if bc_tv and bp_tv and fc_tv and fp_tv:
    df_edge["tv_back_call"]        = df_edge[bc_tv[0]]
    df_edge["tv_front_call"]       = df_edge[fc_tv[0]]
    df_edge["tv_back_put"]         = df_edge[bp_tv[0]]
    df_edge["tv_front_put"]        = df_edge[fp_tv[0]]
    df_edge["call_calendar_edge"]  = df_edge["tv_back_call"] - df_edge["tv_front_call"]
    df_edge["put_calendar_edge"]   = df_edge["tv_back_put"]  - df_edge["tv_front_put"]
    df_edge["total_calendar_edge"] = df_edge["call_calendar_edge"] + df_edge["put_calendar_edge"]

    out = ["snapshot_timestamp",
           "tv_back_call","tv_front_call","call_calendar_edge",
           "tv_back_put", "tv_front_put", "put_calendar_edge",
           "total_calendar_edge"]
    print("Time Value & Calendar Edge - Minute by Minute")
    print("=" * 100)
    print(df_edge[out].to_string(index=False))

    if len(df_edge) >= 2:
        f_e = df_edge["total_calendar_edge"].iloc[0]
        l_e = df_edge["total_calendar_edge"].iloc[-1]
        direction = "COMPRESSED (edge narrowed - structure weakening)" if l_e < f_e else "EXPANDED (edge widened - structure strengthening)"
        print(f"
  Edge at window start : {f_e:.4f}")
        print(f"  Edge at window end   : {l_e:.4f}")
        print(f"  Change               : {l_e - f_e:+.4f}  -> {direction}")
else:
    print("Could not find all 4 time_value columns.")
    print("Available:", piv_tv.columns.tolist())


## Cell 6 - P&L Attribution: What Drove the Profit?

Each leg's contribution measured against the 1:46 PM entry fills. At 3:50 PM shows the bar-chart breakdown of which leg dominated.

In [ ]:
if "diagonal_mark" not in df_m.columns:
    print("Run Cell 4 first.")
else:
    df_attr = df_m.copy()
    df_attr["back_call_contrib"]  =  df_attr["back_call_mark"]  - ENTRY_BACK_CALL
    df_attr["back_put_contrib"]   =  df_attr["back_put_mark"]   - ENTRY_BACK_PUT
    df_attr["front_call_contrib"] =  ENTRY_FRONT_CALL - df_attr["front_call_mark"]
    df_attr["front_put_contrib"]  =  ENTRY_FRONT_PUT  - df_attr["front_put_mark"]

    attr_cols = ["snapshot_timestamp","underlying_price",
                 "back_call_contrib","back_put_contrib",
                 "front_call_contrib","front_put_contrib","unrealized_pnl"]
    print("P&L Attribution vs Entry Fills")
    print("(+) helping  |  (-) hurting")
    print("=" * 105)
    print(df_attr[attr_cols].to_string(index=False))

    at_t = df_attr[df_attr["snapshot_timestamp"] <= TRANSFORM_TIME]
    if len(at_t):
        row = at_t.iloc[-1]
        legs = {
            "Long back call  7500C 6/29": row["back_call_contrib"],
            "Long back put   7300P 6/29": row["back_put_contrib"],
            "Short front call 7500C 6/26": row["front_call_contrib"],
            "Short front put  7300P 6/26": row["front_put_contrib"],
        }
        total = sum(legs.values())
        print(f"
--- Attribution at ~3:50 PM | SPX: {row['underlying_price']:.1f} ---")
        for leg, val in legs.items():
            pct = val / total * 100 if total else 0
            bar = chr(9608) * int(abs(pct) / 4)
            print(f"  {leg:<35} {val:+.2f}  ({pct:5.1f}%)  {bar}")
        print(f"  {'Total P&L vs entry':<35} {total:+.2f}")
        dominant = max(legs, key=lambda k: abs(legs[k]))
        print(f"
  Dominant driver: {dominant}  ({legs[dominant]:+.2f})")
        call_net = row["back_call_contrib"] + row["front_call_contrib"]
        put_net  = row["back_put_contrib"]  + row["front_put_contrib"]
        print(f"
  Call-side net: {call_net:+.2f}")
        print(f"  Put-side net : {put_net:+.2f}")
        if abs(put_net) > abs(call_net):
            print("  -> Put side dominated. SPX moved toward/below 7300.")
        else:
            print("  -> Call side dominated. SPX moved toward/above 7500.")


## Cell 7 - ATM IV Term Structure (Macro Backdrop)

Uses the pre-aggregated `atm_iv_by_expiry` table. Shows whether the broader IV curve was expanding or contracting.

In [ ]:
atm_sql = """
SELECT
    s.snapshot_timestamp,
    s.underlying_price,
    s.vix_value,
    a.expiry_date,
    a.dte,
    a.atm_strike,
    a.atm_call_iv,
    a.atm_put_iv,
    a.atm_avg_iv,
    a.iv_spread_to_front,
    a.iv_ratio_to_front
FROM atm_iv_by_expiry a
JOIN snapshots s ON a.snapshot_id = s.snapshot_id
WHERE
    s.status IN ('COMPLETE','PARTIAL')
    AND s.snapshot_timestamp BETWEEN ? AND ?
    AND a.expiry_date IN (?, ?)
ORDER BY s.snapshot_timestamp, a.expiry_date
"""

df_atm = pd.read_sql_query(
    atm_sql, conn,
    params=(WINDOW_START, WINDOW_END, FRONT_EXPIRY, BACK_EXPIRY)
)
df_atm["snapshot_timestamp"] = pd.to_datetime(df_atm["snapshot_timestamp"])

if df_atm.empty:
    print("No rows in atm_iv_by_expiry for this window.")
else:
    piv = df_atm.pivot_table(
        index=["snapshot_timestamp","underlying_price","vix_value"],
        columns="expiry_date",
        values=["atm_avg_iv","atm_call_iv","atm_put_iv",
                "iv_spread_to_front","iv_ratio_to_front"],
        aggfunc="first"
    )
    piv.columns = ["_".join(str(s) for s in c) for c in piv.columns]
    piv = piv.reset_index().sort_values("snapshot_timestamp")

    print("ATM IV Term Structure - Minute by Minute")
    print("=" * 120)
    pd.set_option("display.max_columns", 25)
    pd.set_option("display.width", 240)
    print(piv.to_string(index=False))

    ratio_cols = [c for c in piv.columns if "iv_ratio" in c]
    if ratio_cols and len(piv) >= 2:
        r_first = piv[ratio_cols[0]].dropna().iloc[0]
        r_last  = piv[ratio_cols[0]].dropna().iloc[-1]
        print(f"
  ATM IV ratio first snapshot: {r_first:.4f}")
        print(f"  ATM IV ratio last snapshot : {r_last:.4f}  ({r_last - r_first:+.4f})")
        if r_last < r_first:
            print("  -> IV ratio FELL: front month IV rose faster. Calendar edge compressed.")
        else:
            print("  -> IV ratio ROSE: back month IV richened. Calendar edge expanded.")


## Cell 8 - Transformation Score (0-100)

| Pillar | Max | Signal |
|--------|-----|--------|
| Profit realized | 40 pts | >= +37% gain ($5 on $13.50) |
| IV edge compression | 35 pts | back/front IV ratio falling toward 1.0 |
| DTE urgency | 25 pts | front expiry within 4 calendar days |

Score >= 70 = candidate | Score >= 85 = act now

In [ ]:
def compute_score(row):
    d = {}
    # Pillar 1: Profit (40 pts)
    mark = row.get("diagonal_mark", np.nan)
    if pd.notna(mark):
        pnl_pct = (mark - ENTRY_NET_DEBIT) / ENTRY_NET_DEBIT
        p_score = min(40.0, max(0.0, pnl_pct * (40.0 / 0.37)))
    else:
        pnl_pct = np.nan
        p_score = 0.0
    d["profit_pct"] = round(pnl_pct * 100, 1) if pd.notna(pnl_pct) else np.nan
    d["profit_pts"] = round(p_score, 1)

    # Pillar 2: IV edge compression (35 pts)
    ratios = [r for r in [row.get("put_iv_ratio"), row.get("call_iv_ratio")]
              if pd.notna(r) and r > 0]
    if ratios:
        avg_r = np.mean(ratios)
        if avg_r > IV_RATIO_THRESHOLD:
            e_score = 0.0
        elif avg_r < 1.0:
            e_score = 35.0
        else:
            e_score = 35.0 * (IV_RATIO_THRESHOLD - avg_r) / (IV_RATIO_THRESHOLD - 1.0)
    else:
        avg_r   = np.nan
        e_score = 0.0
    d["avg_iv_ratio"] = round(avg_r, 4) if pd.notna(avg_r) else np.nan
    d["edge_pts"]     = round(e_score, 1)

    # Pillar 3: DTE urgency (25 pts)
    dte = (date(2026, 6, 26) - date(2026, 6, 23)).days   # 3
    dte_score = 0.0 if dte >= 5 else (25.0 if dte <= 1 else 25.0 * (5 - dte) / 4.0)
    d["dte"]      = dte
    d["dte_pts"]  = round(dte_score, 1)
    d["total_score"] = round(p_score + e_score + dte_score, 1)
    return d

if "diagonal_mark" not in df_m.columns:
    print("Run Cell 4 first.")
else:
    score_df = df_m.apply(compute_score, axis=1, result_type="expand")
    df_score = pd.concat(
        [df_m[["snapshot_timestamp","underlying_price","diagonal_mark","unrealized_pnl"]],
         score_df], axis=1
    )
    cols = ["snapshot_timestamp","underlying_price","diagonal_mark","unrealized_pnl",
            "profit_pct","avg_iv_ratio","profit_pts","edge_pts","dte_pts","total_score"]
    print("Transformation Score - Minute by Minute")
    print(">= 70 = candidate  |  >= 85 = act now")
    print("=" * 100)
    print(df_score[[c for c in cols if c in df_score.columns]].to_string(index=False))

    at_t = df_score[df_score["snapshot_timestamp"] <= TRANSFORM_TIME]
    if len(at_t):
        sc = at_t.iloc[-1]
        print(f"
--- Score at ~3:50 PM ---")
        print(f"  Total       : {sc['total_score']:.1f} / 100")
        print(f"  Profit pillar : {sc.get('profit_pts','n/a')} pts  ({sc.get('profit_pct','n/a')}% return)")
        print(f"  Edge pillar   : {sc.get('edge_pts','n/a')} pts  (avg IV ratio {sc.get('avg_iv_ratio','n/a')})")
        print(f"  DTE pillar    : {sc.get('dte_pts','n/a')} pts  (DTE = {sc.get('dte','n/a')})")


## Cell 9 - Trade Forensics Charts (6 Panels)

Red dashed line marks the 3:50 PM transformation. Chart saved to disk.

In [ ]:
if "diagonal_mark" not in df_m.columns:
    print("Run Cell 4 first.")
else:
    fig = plt.figure(figsize=(17, 15))
    gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.52, wspace=0.35)
    time = df_m["snapshot_timestamp"]

    def vline(ax, label=True):
        ax.axvline(TRANSFORM_TIME, color="red", linestyle="--", linewidth=1.8,
                   label="3:50 PM transform" if label else "_nolegend_")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    # Panel 1: SPX Price
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(time, df_m["underlying_price"], color="black", linewidth=2.2)
    vline(ax1)
    ax1.set_title("SPX Underlying Price  (3:30 - 4:00 PM ET)", fontweight="bold")
    ax1.set_ylabel("SPX")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Panel 2: Diagonal Mark
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(time, df_m["diagonal_mark"], color="royalblue", linewidth=2.2,
             label="Diagonal Mark")
    ax2.axhline(ENTRY_NET_DEBIT, color="gray", linestyle=":", linewidth=1.5,
                label=f"Entry cost (${ENTRY_NET_DEBIT})")
    ax2.axhline(ENTRY_NET_DEBIT + PROFIT_THRESHOLD, color="darkorange",
                linestyle="--", linewidth=1.5, label=f"+${PROFIT_THRESHOLD} threshold")
    ax2.fill_between(time, ENTRY_NET_DEBIT, df_m["diagonal_mark"],
                     where=df_m["diagonal_mark"] > ENTRY_NET_DEBIT,
                     alpha=0.18, color="green", label="Profit zone")
    ax2.fill_between(time, ENTRY_NET_DEBIT, df_m["diagonal_mark"],
                     where=df_m["diagonal_mark"] < ENTRY_NET_DEBIT,
                     alpha=0.18, color="red", label="Loss zone")
    vline(ax2)
    ax2.legend(fontsize=8, ncol=2, loc="upper left")
    ax2.set_title("Diagonal Mark Value vs Time", fontweight="bold")
    ax2.set_ylabel("Mark ($)")
    ax2.grid(True, alpha=0.3)

    # Panel 3: Put Legs
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.plot(time, df_m["back_put_mark"],  color="purple",  linewidth=2.0, label="6/29 7300P (long)")
    ax3.plot(time, df_m["front_put_mark"], color="crimson", linewidth=2.0, label="6/26 7300P (short)")
    ax3.axhline(ENTRY_BACK_PUT,  color="purple",  linestyle=":", alpha=0.5, label=f"Entry {ENTRY_BACK_PUT}")
    ax3.axhline(ENTRY_FRONT_PUT, color="crimson", linestyle=":", alpha=0.5, label=f"Entry {ENTRY_FRONT_PUT}")
    vline(ax3, label=False)
    ax3.legend(fontsize=7)
    ax3.set_title("7300 Put: Front vs Back", fontweight="bold")
    ax3.set_ylabel("Mark ($)")
    ax3.grid(True, alpha=0.3)

    # Panel 4: Call Legs
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.plot(time, df_m["back_call_mark"],  color="navy",   linewidth=2.0, label="6/29 7500C (long)")
    ax4.plot(time, df_m["front_call_mark"], color="sienna", linewidth=2.0, label="6/26 7500C (short)")
    ax4.axhline(ENTRY_BACK_CALL,  color="navy",   linestyle=":", alpha=0.5, label=f"Entry {ENTRY_BACK_CALL}")
    ax4.axhline(ENTRY_FRONT_CALL, color="sienna", linestyle=":", alpha=0.5, label=f"Entry {ENTRY_FRONT_CALL}")
    vline(ax4, label=False)
    ax4.legend(fontsize=7)
    ax4.set_title("7500 Call: Front vs Back", fontweight="bold")
    ax4.set_ylabel("Mark ($)")
    ax4.grid(True, alpha=0.3)

    # Panel 5: IV Spreads
    ax5 = fig.add_subplot(gs[3, 0])
    if "put_iv_spread" in df_m.columns and df_m["put_iv_spread"].notna().any():
        ax5.plot(time, df_m["put_iv_spread"],  color="purple", linewidth=2.0, label="7300P IV spread")
        ax5.plot(time, df_m["call_iv_spread"], color="sienna", linewidth=2.0, label="7500C IV spread")
        ax5.axhline(0, color="black", linewidth=0.8)
        vline(ax5, label=False)
        ax5.legend(fontsize=8)
    else:
        ax5.text(0.5, 0.5, "IV data not captured
(IV column may be null)",
                 transform=ax5.transAxes, ha="center", va="center", color="gray", fontsize=10)
    ax5.set_title("IV Spread: Back - Front per Strike", fontweight="bold")
    ax5.set_ylabel("IV Spread")
    ax5.grid(True, alpha=0.3)

    # Panel 6: Transformation Score
    ax6 = fig.add_subplot(gs[3, 1])
    if "df_score" in dir() and "total_score" in df_score.columns:
        ax6.plot(df_score["snapshot_timestamp"], df_score["total_score"],
                 color="darkgreen", linewidth=2.2, label="Transform Score")
        ax6.axhline(70, color="orange", linestyle="--", linewidth=1.4, label="Candidate (70)")
        ax6.axhline(85, color="red",    linestyle="--", linewidth=1.4, label="Act now (85)")
        ax6.set_ylim(0, 105)
        vline(ax6, label=False)
        ax6.legend(fontsize=8)
    else:
        ax6.text(0.5, 0.5, "Run Cell 8 first", transform=ax6.transAxes,
                 ha="center", va="center", color="gray", fontsize=10)
    ax6.set_title("Transformation Score (0-100)", fontweight="bold")
    ax6.set_ylabel("Score")
    ax6.grid(True, alpha=0.3)

    plt.suptitle(
        "SPX Diagonal Calendar - Trade Forensics  |  Paper Trade 6/23/2026
"
        "Entry 1:46 PM @ $13.50 debit  ->  Transform 3:50 PM @ $19.85 credit  ->  Locked P&L: +$635",
        fontsize=11, fontweight="bold", y=1.005
    )
    out = "trade_forensics_2026_06_23.png"
    plt.savefig(out, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"
Saved -> {out}")


## Cell 10 - Neighboring Strike IV Slope

Compares IV at 7295/7300/7305 and 7495/7500/7505 to confirm whether the IV move
was broad (directional event) or isolated (strike-specific noise).

In [ ]:
nbr_sql = """
SELECT
    s.snapshot_timestamp,
    s.underlying_price,
    o.expiry_date,
    o.strike,
    o.right,
    o.mark,
    o.iv,
    o.delta,
    o.time_value
FROM option_rows o
JOIN snapshots s ON o.snapshot_id = s.snapshot_id
WHERE
    s.status IN ('COMPLETE','PARTIAL')
    AND s.snapshot_timestamp BETWEEN ? AND ?
    AND o.expiry_date IN (?, ?)
    AND (
        (UPPER(o.right) IN ('C','CALL') AND o.strike IN (7495.0, 7500.0, 7505.0))
        OR
        (UPPER(o.right) IN ('P','PUT')  AND o.strike IN (7295.0, 7300.0, 7305.0))
    )
ORDER BY s.snapshot_timestamp, o.expiry_date, o.right, o.strike
"""

df_nbr = pd.read_sql_query(
    nbr_sql, conn,
    params=(WINDOW_START, WINDOW_END, FRONT_EXPIRY, BACK_EXPIRY)
)
df_nbr["snapshot_timestamp"] = pd.to_datetime(df_nbr["snapshot_timestamp"])
df_nbr["right"] = df_nbr["right"].str.upper().str[0]

if df_nbr.empty:
    print("No neighboring strike data found.")
else:
    first_ts = df_nbr["snapshot_timestamp"].min()
    last_ts  = df_nbr["snapshot_timestamp"].max()

    for label, ts in [("FIRST SNAPSHOT (3:30 PM)", first_ts),
                      ("LAST SNAPSHOT  (4:00 PM)", last_ts)]:
        sub = df_nbr[df_nbr["snapshot_timestamp"] == ts].sort_values(
            ["right","expiry_date","strike"])
        print(f"
--- {label} | SPX: {sub['underlying_price'].iloc[0]:.1f} ---")
        print(sub[["expiry_date","right","strike","mark","iv","delta","time_value"]].to_string(index=False))

    print("
--- IV Change (Last - First) per contract ---")
    iv_f   = df_nbr[df_nbr["snapshot_timestamp"]==first_ts].set_index(["expiry_date","right","strike"])["iv"]
    iv_l   = df_nbr[df_nbr["snapshot_timestamp"]==last_ts ].set_index(["expiry_date","right","strike"])["iv"]
    iv_chg = (iv_l - iv_f).reset_index()
    iv_chg.columns = ["expiry_date","right","strike","iv_change"]
    iv_chg["signal"] = iv_chg["iv_change"].apply(
        lambda x: "EXPANDED" if x > 0.002 else ("CONTRACTED" if x < -0.002 else "flat")
    )
    print(iv_chg.to_string(index=False))

    print("
--- IV slope across strikes at last snapshot ---")
    slope = df_nbr[df_nbr["snapshot_timestamp"]==last_ts].pivot_table(
        index=["expiry_date","right"], columns="strike", values="iv", aggfunc="first"
    )
    print(slope.to_string())


## Cell 11 - Raw SQL Explorer

Edit the query and re-run freely. Default shows the back-month 7300P minute by minute.

In [ ]:
custom_sql = """
SELECT
    s.snapshot_timestamp,
    s.underlying_price,
    s.vix_value,
    o.expiry_date,
    o.strike,
    o.right,
    o.bid,
    o.ask,
    o.mark,
    o.iv,
    o.delta,
    o.vega,
    o.time_value
FROM option_rows o
JOIN snapshots s ON o.snapshot_id = s.snapshot_id
WHERE
    s.snapshot_timestamp BETWEEN ? AND ?
    AND o.expiry_date = ?
    AND UPPER(o.right) IN ('P','PUT')
    AND o.strike = 7300.0
ORDER BY s.snapshot_timestamp
"""

df_custom = pd.read_sql_query(
    custom_sql, conn,
    params=(WINDOW_START, WINDOW_END, BACK_EXPIRY)
)
df_custom["snapshot_timestamp"] = pd.to_datetime(df_custom["snapshot_timestamp"])
print(f"Rows: {len(df_custom)}")
print(df_custom.to_string(index=False))


## Cell 12 - Summary & Close

In [ ]:
conn.close()
print("Database connection closed.
")
print("=" * 62)
print("  TRADE FORENSICS - FINAL SUMMARY")
print("=" * 62)
print(f"  Entry (1:46 PM) net debit    : ${ENTRY_NET_DEBIT}")
print(f"  Transform (3:50 PM) credit   : ${TRANSFORM_CREDIT}")
print(f"  Locked P&L                   : +${LOCKED_PNL:.2f}  (+${LOCKED_PNL*100:.0f} per contract)")
print()
print("  Resulting Iron Condor (6/26 expiry):")
print("    Short 7500C / Long 7505C  bear call spread  width $5")
print("    Short 7300P / Long 7295P  bull put spread   width $5")
print("    Max loss   : ~$135")
print("    Max profit : ~$635")
print()
if "df_m" in dir() and "diagonal_mark" in df_m.columns and len(df_m):
    at_t = df_m[df_m["snapshot_timestamp"] <= TRANSFORM_TIME]
    if len(at_t):
        r = at_t.iloc[-1]
        print(f"  Diagonal mark at ~3:50 PM  : {r['diagonal_mark']:.4f}")
        print(f"  Unrealized P&L at ~3:50    : +{r['unrealized_pnl']:.4f}")
if "df_score" in dir() and "total_score" in df_score.columns and len(df_score):
    at_t2 = df_score[df_score["snapshot_timestamp"] <= TRANSFORM_TIME]
    if len(at_t2):
        print(f"  Transform score at ~3:50   : {at_t2.iloc[-1]['total_score']:.1f} / 100")
print()
print("  Chart saved: trade_forensics_2026_06_23.png")
print()
print("  Next: Collect 4 more trading days, then calibrate")
print("  the transformation score thresholds from real data.")
